# Task 1: Image Feature Extraction
Extract image features from the Flickr8k dataset using VGG16 and save them to `features.pkl`.

In [1]:
!pip install --upgrade kaggle tensorflow pillow tqdm matplotlib

  Using cached tensorflow-2.13.1-cp38-cp38-win_amd64.whl.metadata (2.6 kB)
INFO: pip is looking at multiple versions of tensorflow to determine which version is compatible with other requirements. This could take a while.


In [2]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input


## Step 1: Download Flickr8k Dataset
Make sure `kaggle.json` exists at `C:\Users\<you>\.kaggle\kaggle.json` with your Kaggle credentials before running this cell. The dataset is only downloaded once, subsequent runs will skip this step.

In [3]:
# Only download if the Images folder doesn't already exist
# kaggle.json must exist at C:\Users\<you>\.kaggle\kaggle.json
if os.path.exists("./flickr8k/Images"):
    print("Flickr8k already downloaded — skipping download.")
else:
    !kaggle datasets download -d adityajn105/flickr8k --unzip -p ./flickr8k
    print("Download complete!")

Flickr8k already downloaded — skipping download.


In [4]:
images_dir = "./flickr8k/Images"
captions_file = "./flickr8k/captions.txt"

image_files = [f for f in os.listdir(images_dir) if f.endswith(".jpg")]
print(f"Total images found: {len(image_files)}")
print(f"Sample filenames: {image_files[:5]}")

Total images found: 8091
Sample filenames: ['1000268201_693b08cb0e.jpg', '1001773457_577c3a7d70.jpg', '1002674143_1b742ab4b8.jpg', '1003163366_44323f5815.jpg', '1007129816_e794419615.jpg']


## Step 2: Load VGG16 Model
VGG16 is pre-trained on ImageNet. We remove the top classification layer (`include_top=False`) and use it purely as a feature extractor. The output is a (7, 7, 512) feature map which we flatten to a **25088-dim vector** per image.

In [5]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False
print("VGG16 loaded successfully!")
print(f"Output shape: {base_model.output_shape}")

VGG16 loaded successfully!
Output shape: (None, 7, 7, 512)


## Step 3: Extract Features
Each image is resized to 224×224 and preprocessed for VGG16. Images are processed in batches for efficiency. The output feature map (7×7×512) is flattened to a **25088-dim vector** per image.

In [6]:
BATCH_SIZE = 32

def extract_features_batch(image_files, images_dir, model, batch_size=BATCH_SIZE):
    features = {}
    for i in tqdm(range(0, len(image_files), batch_size), desc="Extracting features"):
        batch_names = image_files[i:i + batch_size]
        batch_arrays = []
        valid_names = []
        for img_name in batch_names:
            try:
                img = Image.open(os.path.join(images_dir, img_name)).convert("RGB")
                img = img.resize((224, 224))
                img_array = preprocess_input(np.array(img, dtype=np.float32))
                batch_arrays.append(img_array)
                valid_names.append(img_name)
            except Exception as e:
                print(f"Skipping {img_name}: {e}")
        batch_preds = model.predict(np.stack(batch_arrays), verbose=0)
        for name, feat in zip(valid_names, batch_preds):
            features[name] = feat.flatten()  # Shape: (25088,)
    return features

In [7]:
if os.path.exists("features.pkl"):
    print("features.pkl already exists — skipping extraction.")
    print("Delete features.pkl and re-run if you want to regenerate.")
else:
    features = extract_features_batch(image_files, images_dir, base_model)
    print(f"\nFeature extraction complete!")
    print(f"Total images processed: {len(features)}")
    print(f"Feature vector shape: {features[image_files[0]].shape}")

Extracting features: 100%|██████████| 253/253 [09:37<00:00,  2.28s/it]


Feature extraction complete!
Total images processed: 8091
Feature vector shape: (25088,)


## Step 4: Save Features
Save the features dictionary to `features.pkl`. This file will be loaded by Task 3 for model training.

In [8]:
with open("features.pkl", "wb") as f:
    pickle.dump(features, f)

print("Features saved to 'features.pkl'")

Features saved to 'features.pkl'
